In [2]:
# =============================================================================
# 0. Import libraries
# =============================================================================

import pandas as pd

In [3]:
# =============================================================================
# 1. Define input files and model names
# =============================================================================

file_info = [
    {
        "path": "svm_10fold_instance_predictions.csv",
        "model": "SVM"
    },
    {
        "path": "bert_10fold_instance_predictions.csv",
        "model": "BERT"
    },
    {
        "path": "zero_shot_10fold_instance_predictions.csv",
        "model": "Llama Zero-shot"
    },
    {
        "path": "few_shot_10fold_instance_predictions.csv",
        "model": "Llama Few-shot"
    }
]

In [4]:
# =============================================================================
# 2. Define a function to calculate confusion matrix components and error rates
# =============================================================================

def compute_error_summary(
    df,
    model_name
):
    """
    Calculate TP, FP, FN, TN, false positive rate,
    and false negative rate for one model.

    Class 1 = Government
    Class 0 = Non-government
    """

    # Ensure that true and predicted classes are numeric.
    df = df.copy()

    df["True_Class"] = (
        df["True_Class"]
        .astype(int)
    )

    df["Predicted_Class"] = (
        df["Predicted_Class"]
        .astype(int)
    )

    y_true = df["True_Class"]
    y_pred = df["Predicted_Class"]
    
    # -------------------------------------------------------------------------
    # Confusion matrix components
    # -------------------------------------------------------------------------

    tp = (
        (y_true == 1)
        & (y_pred == 1)
    ).sum()

    fp = (
        (y_true == 0)
        & (y_pred == 1)
    ).sum()

    fn = (
        (y_true == 1)
        & (y_pred == 0)
    ).sum()

    tn = (
        (y_true == 0)
        & (y_pred == 0)
    ).sum()

    # -------------------------------------------------------------------------
    # Error rates
    # -------------------------------------------------------------------------

    # False Positive Rate:
    # proportion of actual non-government organizations
    # incorrectly classified as government.
    fp_rate = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else 0
    )

    # False Negative Rate:
    # proportion of actual government organizations
    # incorrectly classified as non-government.
    fn_rate = (
        fn / (fn + tp)
        if (fn + tp) > 0
        else 0
    )

    return {
        "Model": model_name,
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
        "FP_Rate": fp_rate,
        "FN_Rate": fn_rate
    }

In [5]:
# =============================================================================
# 3. Read all prediction files and calculate error summaries
# =============================================================================

summary_rows = []

for item in file_info:

    df = pd.read_csv(
        item["path"],
        encoding="utf-8-sig"
    )

    # Keep only rows with valid true and predicted classes.
    df = df.dropna(
        subset=[
            "True_Class",
            "Predicted_Class"
        ]
    ).copy()

    summary = compute_error_summary(
        df=df,
        model_name=item["model"]
    )

    summary_rows.append(
        summary
    )

In [6]:
# =============================================================================
# 4. Create the final error analysis table
# =============================================================================

error_summary_df = pd.DataFrame(
    summary_rows
)

error_summary_df = error_summary_df[
    [
        "Model",
        "TP",
        "FP",
        "FN",
        "TN",
        "FP_Rate",
        "FN_Rate"
    ]
]

In [7]:
# =============================================================================
# 5. Round the error rates
# =============================================================================

error_summary_df[
    [
        "FP_Rate",
        "FN_Rate"
    ]
] = error_summary_df[
    [
        "FP_Rate",
        "FN_Rate"
    ]
].round(4)


In [8]:
# =============================================================================
# 6. Display the table
# =============================================================================

display(
    error_summary_df
)


,Model,TP,FP,FN,TN,FP_Rate,FN_Rate
0,SVM,207,54,48,191,0.2204,0.1882
1,BERT,219,29,36,216,0.1184,0.1412
2,Llama Zero-shot,224,19,31,226,0.0776,0.1216
3,Llama Few-shot,219,13,36,232,0.0531,0.1412


In [9]:
# =============================================================================
# 7. Save the table as a CSV file
# =============================================================================

OUTPUT_FILE = "table_error_analysis.csv"

error_summary_df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"Error analysis table saved to: "
    f"{OUTPUT_FILE}"
)

Error analysis table saved to: table_error_analysis.csv
